In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

## Getting the Raw data in a DataFrame

In [5]:
df1 = pd.read_json("MyData/StreamingHistory0.json")
df2 = pd.read_json("MyData/StreamingHistory1.json")
df3 = pd.read_json("MyData/StreamingHistory2.json")

In [6]:
df = pd.concat([df1, df2, df3])

In [7]:
df.reset_index(drop=True, inplace=True)

In [9]:
df.to_csv("MyData/StreamingHistory.csv", index=False)

In [10]:
len(df)

22804

## Some Preprocessing

### Minutes Played

In [11]:
df["Minutes Played"] = np.round(df["msPlayed"]/60000, 2)

In [12]:
df.head()

,endTime,artistName,trackName,msPlayed,Minutes Played
0,2021-12-23 15:37,Taylor Swift,Untouchable (Taylor’s Version),186569,3.11
1,2021-12-24 02:37,Adele,Easy On Me,224694,3.74
2,2021-12-24 02:42,Adele,Someone Like You,285240,4.75
3,2021-12-24 02:47,Adele,Love In The Dark,285935,4.77
4,2021-12-24 02:51,Adele,Rolling in the Deep,228093,3.80


### Adding Day, Week and Month

In [13]:
df["endTime"] = pd.to_datetime(df["endTime"])

In [14]:
df['Day'] = df['endTime'].dt.day
df["Week"] = df['endTime'].dt.week
df["Month"] = df["endTime"].dt.month

/tmp/ipykernel_7220/2036370107.py:2: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated. Please use Series.dt.isocalendar().week instead.
  df["Week"] = df['endTime'].dt.week


In [15]:
df.head()

,endTime,artistName,trackName,msPlayed,Minutes Played,Day,Week,Month
0,2021-12-23 15:37:00,Taylor Swift,Untouchable (Taylor’s Version),186569,3.11,23,51,12
1,2021-12-24 02:37:00,Adele,Easy On Me,224694,3.74,24,51,12
2,2021-12-24 02:42:00,Adele,Someone Like You,285240,4.75,24,51,12
3,2021-12-24 02:47:00,Adele,Love In The Dark,285935,4.77,24,51,12
4,2021-12-24 02:51:00,Adele,Rolling in the Deep,228093,3.80,24,51,12


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22804 entries, 0 to 22803
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   endTime         22804 non-null  datetime64[ns]
 1   artistName      22804 non-null  object        
 2   trackName       22804 non-null  object        
 3   msPlayed        22804 non-null  int64         
 4   Minutes Played  22804 non-null  float64       
 5   Day             22804 non-null  int64         
 6   Week            22804 non-null  int64         
 7   Month           22804 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(4), object(2)
memory usage: 1.4+ MB


### Dropping Unnecessary Columns and saving the data

In [17]:
df.drop("msPlayed", axis=1, inplace=True)

In [18]:
extras_df = df[(df["Minutes Played"]>0.5)].reset_index(drop=True)

In [19]:
len(extras_df)

20529

In [20]:
extras_df.to_csv("MyData/StreamingHistoryExtras.csv", index=False)

## Grouping Data

In [32]:
extras_df = pd.read_csv("MyData/StreamingHistoryExtras.csv")

### By Artist

In [21]:
all_artists = extras_df["artistName"].unique()
len(all_artists)

91

We'll drop all artists that have have less than  hour of runtime.

In [23]:
df_grouped_by_artists = extras_df.groupby("artistName").sum(numeric_only=True)["Minutes Played"]
len(df_grouped_by_artists)

91

In [24]:
top_artists = df_grouped_by_artists[df_grouped_by_artists>60].sort_values(ascending=False).index.tolist()
len(top_artists)

38

In [25]:
top_artists

['Taylor Swift',
 'Adele',
 'Céline Dion',
 'Ed Sheeran',
 'Lata Mangeshkar',
 'Camila Cabello',
 'Lady Gaga',
 'Billie Eilish',
 'Alan Walker',
 'Shawn Mendes',
 'Maren Morris',
 'Beyoncé',
 'Lea Salonga',
 'Britney Spears',
 'Coldplay',
 'Sia',
 'Selena Gomez',
 'Zac Efron',
 'PUBLIC',
 'Idina Menzel',
 'Christina Perri',
 'Queen',
 'Miranda Lambert',
 'Dua Lipa',
 'Ryan Hurd',
 'Lewis Capaldi',
 'Mohammed Rafi',
 'Mandy Moore',
 'Justin Bieber',
 'Bhupen Hazarika',
 'Harry Styles',
 'Lorne Balfe',
 'Lauv',
 'Mukesh',
 'Kishore Kumar',
 'Susan Egan',
 'Hemlata',
 'J Balvin']

In [37]:
artists_dict = {}
for artist in top_artists:
    artists_dict[artist] = extras_df[extras_df["artistName"]==artist]

In [38]:
for artist in artists_dict.keys():
    artist_df = artists_dict[artist].reset_index(drop=True)
    artist_df.to_csv(f"MyData\{artist}.csv", index=False)

### By Month

In [46]:
month_dict = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}
for i in range(1,13):
    month_df = extras_df[extras_df["Month"]==i].reset_index(drop=True)
    month_df.to_csv(f"MyData/{month_dict[i]}.csv", index=False)

In [49]:
extras_df[extras_df["Month"]==3]

,endTime,artistName,trackName,Day,Week,Month,Minutes Played
3801,2021-03-01 01:56:00,Alan Walker,Darkside,1,9,3,1.36
3802,2021-03-01 02:01:00,Ed Sheeran,Perfect,1,9,3,4.39
3803,2021-03-01 02:05:00,Ed Sheeran,Shape of You,1,9,3,3.90
3804,2021-03-01 02:08:00,Justin Bieber,Anyone,1,9,3,3.18
3805,2021-03-01 02:11:00,Shawn Mendes,Where Were You In The Morning?,1,9,3,3.34
...,...,...,...,...,...,...,...
5918,2021-03-31 14:16:00,Alan Walker,Darkside,31,13,3,3.53
5919,2021-03-31 14:20:00,Taylor Swift,Fearless,31,13,3,4.03
5920,2021-03-31 14:24:00,Céline Dion,I'm Alive,31,13,3,3.50
5921,2021-03-31 14:29:00,Taylor Swift,evermore (feat. Bon Iver),31,13,3,5.07
